In [ ]:
from pathlib import Path
import sys, json, html
from IPython.display import display, HTML
ROOT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
# 不对运行中的Dataset热替换算子：发现旧内核时停止，重启后再读取checkpoint。
def _assert_current_kernel():
    import os
    try:
        shell = get_ipython()
    except NameError:
        return  # CLI是独立Python进程。
    if shell is None or not hasattr(shell, 'kernel'):
        return
    boot = next(int(line.split()[1]) for line in Path('/proc/stat').read_text().splitlines() if line.startswith('btime '))
    started = boot + int(Path('/proc/self/stat').read_text().rsplit(')', 1)[1].split()[19]) / os.sysconf('SC_CLK_TCK')
    changed = []
    for name, module in tuple(sys.modules.items()):
        if name.startswith(('curation.', 'demiflow.')):
            filename = getattr(module, '__file__', None)
            if filename and Path(filename).is_file() and Path(filename).stat().st_mtime > started:
                changed.append(name)
    if changed:
        raise RuntimeError('当前内核启动后算子代码已更新，请重启内核并重新执行初始化；禁止新旧算子混用。涉及：' + ', '.join(changed[:6]))
_assert_current_kernel()
from curation.v4.ops.filter_document_blocks import FilterDocumentBlocks
from curation.v4.ops.select_source_records import SelectSourceRecords
from curation.v4.ops.cross_batch import BatchRelationshipReviews, ApplyRelationshipReviews
from demiflow.standalone import local_data
from curation.v4.contracts import snapshot, immutable, digest, source_code, runtime_version
from curation.v4.pipeline import DEFAULT
from curation.v4.ops.image_filter import (IMAGE_FILTER_DEFAULTS, RecordPrimaryImageSelection, PrepareImageReview, ApplyConfirmedImageSelection)
from curation.v4.image_filter_runtime import image_prompt_data, review_needed, save_image_filter_policy, validate_material_reuse
from curation.v4.local_review_service import image_review_service
from curation.v4.ops.dataset_operators import (
    ConceptFromRecord, DocumentFromRecord, ImageFromRecord, SelectConcept, MaterialLinks,
    ReadDocument, CleanDocument, CheckImage, CountMaterial, NestMaterial,
    merge_concept, distinct, fill_material_counts, model_input)
from curation.v4.ops.prompt_operators import PrepareIdentity, ApplyIdentity
from curation.v4.ops.prompt_config import knowledge_prompt_pack, prompt_execution_options, save_prompt_config
from curation.v4.ops.source_blocks import BuildSourceBlocks, BatchSourceBlocks, ApplyBlockSelection, merge_block_decisions
from curation.v4.ops.multimodal import SelectAvailableImages, BatchImageSelection, ApplyImageSelection, merge_image_decisions, SelectRelatedMaterials
from curation.v4.ops.material_routing import PrepareRoutingMaterials, RawPassageRows, EncodeImageTextMaterials, BuildRoutedJointRequest
from curation.v4.ops.paragraph_similarity import EmbedParagraphBatch, ParagraphRows
from curation.v4.ops.paragraphs import ApplyParagraphs, ApplyParagraphReview, SelectRetainedParagraphs
from curation.v4.ops.token_routing import RouteByTokenBudget
from curation.v4.ops.paragraph_pipeline import RouteConceptMaterials, PrepareVerifiedParagraphs, BuildLocalMergeGroups, ApplyLocalIntegration, SourceCatalog, FormatTopicArticle, FinalKnowledgeRecord
from curation.v4.ops.cross_batch import PlanCrossBatchReview, ApplyCrossBatchReview
from curation.v4.ops.paragraph_merge import ApplyParagraphMerge
from curation.v4.ops.topic_quality import PrepareTopicRepairs, ApplyTopicRepairs, RetainReviewedTopics, PrepareTopicVerification
from curation.v4.ops.topic_articles import TopicRows

DATASET = ROOT / 'datasets/demiwtg'
RUN = ROOT / 'state/curation/v4/glass_operator_dual_image_v1'
CONCEPT = '玻璃棒'
IDS = ['legacy:' + CONCEPT]
GROUP_SIZE = 256
# 本轮三个概念的明确身份范围；扩量时从概念资料确定，勿按名称猜物种。
IMAGE_IDENTITY_DEFINITIONS = {'legacy:OK手势': '拇指和食指相触成环、其余手指伸展或放松的手势；相关图解和实际使用场景也可保留。', 'legacy:玻璃棒': '实验室中用于搅拌、引流等的实心玻璃棒；同属实验器材不自动属于目标。', 'legacy:白花芍药': '植物学物种 Paeonia sterniana；泛指白色芍药花或其他栽培品种不自动认证为这一物种。身份不确定时保留不确定性。'}
config = {**DEFAULT, **IMAGE_FILTER_DEFAULTS, 'image_identity_definitions': IMAGE_IDENTITY_DEFINITIONS, 'text_mode':'multimodal', 'body_only':True, 'max_calls':None,
          'max_output_tokens':16384, 'temperature':0, 'timeout_s':900,
          'block_unit_chars':1800, 'block_batch_chars':8000,
          'joint_input_tokens':32768, 'image_batch_size':4,
          'text_embedding_model':str(ROOT.parent / 'models/Qwen3-Embedding-0.6B'),
          'image_embedding_model':str(ROOT.parent / 'models/siglip2-base-patch16-224')}
tables = RUN / 'datasets'
knowledge_run = RUN / 'knowledge'

def show(ds, columns=None, n=100):
    from curation.notebook_image_preview import show as preview
    return preview(ds, columns=columns, n=n, run=RUN, dataset=DATASET)


In [ ]:
RUN = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg/state/curation/notebook_runtime_check_v1/glass_step32')
tables = RUN / "datasets"
knowledge_run = RUN / "knowledge"

In [ ]:
concept_source = {'kind':'legacy_concepts', **snapshot(DATASET / 'meta/concepts.json')}
document_source = {'kind':'legacy_docs', **snapshot(DATASET / 'meta/docs.jsonl')}
image_source = {'kind':'legacy_images', **snapshot(DATASET / 'meta/images.jsonl')}
notebook = json.loads((ROOT / 'curation/v4/glass_operator_debug.ipynb').read_text())
manifest = {'sources':[concept_source, document_source, image_source],
            'ids':IDS, 'group_size':GROUP_SIZE, 'config':config,
            'code':source_code(), 'runtime':runtime_version(),
            'cells':[''.join(c['source']) for c in notebook['cells'] if c['cell_type']=='code']}
from curation.notebook_image_preview import preserve_display_version
manifest = preserve_display_version(RUN, manifest)
immutable(RUN / 'manifest.json', manifest)
version = digest(manifest)
pack, prompt_text = knowledge_prompt_pack(config)
options = prompt_execution_options(RUN, config)
save_prompt_config(RUN, prompt_text, options)
data = local_data(prompt_packs={'knowledge.yaml':pack}, prompt_options=options)

# 每个后续步骤执行前核对运行中代码/依赖/配置，禁止沿用旧version写新结果。
import copy
_frozen_code, _frozen_runtime = source_code(), runtime_version()
_frozen_config, _frozen_run = copy.deepcopy(config), RUN

def _assert_run_current():
    _assert_current_kernel()
    if RUN != _frozen_run or config != _frozen_config or source_code() != _frozen_code or runtime_version() != _frozen_runtime:
        raise RuntimeError('本次冻结后代码、依赖、配置或RUN已变化；请重启内核并使用新RUN，不得混用旧checkpoint。')


In [ ]:
blocks = data.read_json(str(ROOT / "state/curation/v4/glass_operator_dual_image_v1/datasets/blocks.jsonl"))
show = lambda *args, **kwargs: None

In [ ]:
_assert_run_current()
image_requests = await blocks.flat_map(BatchImageSelection(config['image_batch_size'], config['image_identity_definitions'], neutral=True)).checkpoint_async(tables / 'image_requests.jsonl', version=version)
show(image_requests, columns=['case_id', 'image_prompt', 'pixel_images'])


In [ ]:
rows = image_requests.take_all()
assert len(rows) == 16
assert sum(len(r['image_prompt']['image_ids']) for r in rows) == 63
assert all('metadata' not in r['image_prompt'] for r in rows)
config['joint_input_tokens'] = 1
try:
    _assert_run_current()
except RuntimeError:
    print('PASS: changed config blocked before operator execution')
else:
    raise AssertionError('config change was not blocked')
print('PASS: actual glass step32, 63 images / 16 requests, zero model calls')